# Notebook per gestire un addestramento

In [1]:
import subprocess
import os
import time
import ipywidgets as widgets
import threading

In [2]:
train_py_path = "train.py"
log_dir = "logs"
log_path = os.path.join(log_dir, f"train_{int(time.time())}.log")
tb_log_dir = "tb_logs"
os.makedirs(log_dir, exist_ok=True)
os.makedirs(tb_log_dir, exist_ok=True)

def run_training_and_tensorboard():
    training_command = "tmux new-session -d -s smash_training 'python3 {}'".format(train_py_path)
    tensorboard_command = "tmux new-session -d -s smash_tensorboard 'tensorboard --logdir {} --port 6006' --bind_all".format(tb_log_dir)
    subprocess.run(training_command, shell=True, check=True)
    #subprocess.run(tensorboard_command, shell=True, check=True)

In [3]:
run_training_and_tensorboard()

In [ ]:
worker_log_template = "/tmp/melee_worker_{rank}.log"

def get_num_workers():
    # Count the number of worker log files
    return len([f for f in os.listdir("/tmp") if f.startswith("melee_worker_") and f.endswith(".log")])

num_workers = 1 #get_num_workers()


output = widgets.Output()

def update_output(change):
    with output:
        output.clear_output()
        print(f"Number of active workers: {num_workers}")
        for rank in range(num_workers):
            log_file = worker_log_template.format(rank=rank)
            if os.path.exists(log_file):
                print(f"\n--- Worker {rank} Log ---")
                with open(log_file, 'r') as f:
                    lines = f.readlines()
                    # Print the last 10 lines of the log
                    for line in lines[-10:]:
                        print(line, end='')

display(output)

output_updater = threading.Thread(target=lambda: [update_output(None) or time.sleep(5) for _ in iter(int, 1)], daemon=True)
output_updater.start()


Output()